In [0]:

------  View full Dataset------------
Select *
From workspace.default.user_profile;

Select *
From workspace.default.Viewership;
---------------------------------

-------Data Exploration-----------
------------------------------------
----------Checking the amount of Rows----------
SELECT COUNT(*) AS total_rows
FROM  workspace.default.user_profile;

SELECT COUNT(*) AS total_rows
FROM  workspace.default.Viewership;

------------Checking for Duplicates ---------------------
SELECT Distinct UserID
FROM workspace.default.user_profile;

SELECT Distinct UserID0
FROM workspace.default.viewership;
-----------------------------------------------
------Checking for Nulls --------
Select*
FROM workspace.default.viewership
WHERE Channel2 IS NULL
  AND `Duration 2` IS NULL
  AND userid4 IS NULL
  AND RecordDate2 is Null;

Select *
From workspace.default.user_profile
WHERE Email IS NULL
  AND Gender IS NULL
  AND Race IS NULL
  AND Age IS NULL
  AND Province IS NULL
  AND `Social Media Handle` IS NULL;
--------------------------------------------------------------
-------------fixing text format--------------------
SELECT 
    TRIM(UPPER(UserID)) AS UserID
FROM workspace.default.user_profile;

----------Fixing Date and Duration--------
SELECT 
    CAST(RecordDate2 AS DATE) AS Record_Date
FROM viewership;

SELECT 
    `Duration 2` AS Duration
    FROM viewership;
    ----------------------------------------
    ----------Left join 2 tables---------

SELECT 
    u.Race,
    u.Age,
    u.Gender,
    u.Province,
    v.Channel2,
    v.RecordDate2,
    v.`Duration 2`
FROM viewership v
LEFT JOIN user_profile u
ON v.UserID0 = u.UserID;

----------------------------------------------

-------------Seprating my date and time----------
SELECT 
  CAST(RecordDate2 AS DATE) AS date_part,
  DATE_FORMAT(RecordDate2, 'HH:mm:ss') AS `Time Bucket`
FROM workspace.default.viewership;

------Converting of UTC time zone to SA Time----
SELECT 
    RecordDate2,
    RecordDate2 + INTERVAL 2 HOURS AS SA_Time
FROM viewership;

-------------------------------------------------
----------SESSION METRICS ANALYSIS---------------
-------------------------------------------------

-- Overall Session Metrics
SELECT 
    COUNT(*) AS total_sessions,
    COUNT(DISTINCT UserID0) AS unique_viewers,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT UserID0), 2) AS avg_sessions_per_user,
    -- Convert Duration 2 from timestamp to seconds (time since 1899-12-31 midnight)
    ROUND(AVG(UNIX_TIMESTAMP(`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')), 2) AS avg_duration_seconds,
    ROUND(SUM(UNIX_TIMESTAMP(`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 3600, 2) AS total_viewing_hours,
    MIN(RecordDate2) AS first_session_date,
    MAX(RecordDate2) AS last_session_date
FROM workspace.default.viewership;

-- Top 10 Most Popular Channels by Session Count
SELECT 
    Channel2,
    COUNT(*) AS session_count,
    COUNT(DISTINCT UserID0) AS unique_viewers,
    ROUND(AVG(UNIX_TIMESTAMP(`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes
FROM workspace.default.viewership
WHERE Channel2 IS NOT NULL
GROUP BY Channel2
ORDER BY session_count DESC
LIMIT 10;

-- Sessions by Hour of Day (SA Time)
SELECT 
    HOUR(RecordDate2 + INTERVAL 2 HOURS) AS hour_of_day,
    COUNT(*) AS session_count,
    ROUND(AVG(UNIX_TIMESTAMP(`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes
FROM workspace.default.viewership
GROUP BY HOUR(RecordDate2 + INTERVAL 2 HOURS)
ORDER BY hour_of_day;

-- Sessions by Day of Week (SA Time)
SELECT 
    DAYOFWEEK(RecordDate2 + INTERVAL 2 HOURS) AS day_of_week,
    CASE DAYOFWEEK(RecordDate2 + INTERVAL 2 HOURS)
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_name,
    COUNT(*) AS session_count,
    ROUND(AVG(UNIX_TIMESTAMP(`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes
FROM workspace.default.viewership
GROUP BY DAYOFWEEK(RecordDate2 + INTERVAL 2 HOURS)
ORDER BY day_of_week;

-- User Engagement: Sessions per User Distribution
SELECT 
    sessions_per_user,
    COUNT(*) AS user_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM (
    SELECT 
        UserID0,
        COUNT(*) AS sessions_per_user
    FROM workspace.default.viewership
    WHERE UserID0 IS NOT NULL
    GROUP BY UserID0
)
GROUP BY sessions_per_user
ORDER BY sessions_per_user;

-- Session Duration Distribution (in minutes)
SELECT 
    CASE 
        WHEN duration_minutes < 1 THEN '0-1 min'
        WHEN duration_minutes < 5 THEN '1-5 min'
        WHEN duration_minutes < 15 THEN '5-15 min'
        WHEN duration_minutes < 30 THEN '15-30 min'
        WHEN duration_minutes < 60 THEN '30-60 min'
        ELSE '60+ min'
    END AS duration_bucket,
    COUNT(*) AS session_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM (
    SELECT 
        (UNIX_TIMESTAMP(`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60 AS duration_minutes
    FROM workspace.default.viewership
)
GROUP BY duration_bucket
ORDER BY 
    CASE duration_bucket
        WHEN '0-1 min' THEN 1
        WHEN '1-5 min' THEN 2
        WHEN '5-15 min' THEN 3
        WHEN '15-30 min' THEN 4
        WHEN '30-60 min' THEN 5
        WHEN '60+ min' THEN 6
    END;

-------------------------------------------------
----------USER-LEVEL AGGREGATIONS----------------
-------------------------------------------------

-- Top 20 Most Active Users by Sessions
SELECT 
    v.UserID0,
    u.Age,
    u.Gender,
    u.Race,
    u.Province,
    COUNT(*) AS total_sessions,
    ROUND(SUM(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 3600, 2) AS total_viewing_hours,
    ROUND(AVG(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_session_minutes,
    COUNT(DISTINCT v.Channel2) AS unique_channels_watched
FROM workspace.default.viewership v
LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
WHERE v.UserID0 IS NOT NULL
GROUP BY v.UserID0, u.Age, u.Gender, u.Race, u.Province
ORDER BY total_sessions DESC
LIMIT 20;

-- User Viewing Profile: Favorite Channel per User (Top 100 users)
WITH user_channel_counts AS (
    SELECT 
        UserID0,
        Channel2,
        COUNT(*) AS channel_sessions,
        ROW_NUMBER() OVER (PARTITION BY UserID0 ORDER BY COUNT(*) DESC) AS channel_rank
    FROM workspace.default.viewership
    WHERE UserID0 IS NOT NULL AND Channel2 IS NOT NULL
    GROUP BY UserID0, Channel2
),
user_totals AS (
    SELECT 
        v.UserID0,
        u.Age,
        u.Gender,
        u.Race,
        COUNT(*) AS total_sessions,
        ROUND(SUM(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 3600, 2) AS total_hours
    FROM workspace.default.viewership v
    LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
    WHERE v.UserID0 IS NOT NULL
    GROUP BY v.UserID0, u.Age, u.Gender, u.Race
)
SELECT 
    ut.UserID0,
    ut.Age,
    ut.Gender,
    ut.Race,
    ut.total_sessions,
    ut.total_hours,
    ucc.Channel2 AS favorite_channel,
    ucc.channel_sessions AS favorite_channel_sessions,
    ROUND(ucc.channel_sessions * 100.0 / ut.total_sessions, 2) AS favorite_channel_percentage
FROM user_totals ut
INNER JOIN user_channel_counts ucc ON ut.UserID0 = ucc.UserID0
WHERE ucc.channel_rank = 1
ORDER BY ut.total_sessions DESC
LIMIT 100;

-------------------------------------------------
------CHANNEL POPULARITY BY DEMOGRAPHICS---------
-------------------------------------------------

-- Top Channels by Race
SELECT 
    u.Race,
    v.Channel2,
    COUNT(*) AS session_count,
    COUNT(DISTINCT v.UserID0) AS unique_viewers,
    ROUND(AVG(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes,
    ROW_NUMBER() OVER (PARTITION BY u.Race ORDER BY COUNT(*) DESC) AS channel_rank
FROM workspace.default.viewership v
LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
WHERE v.Channel2 IS NOT NULL AND u.Race IS NOT NULL AND u.Race != 'None'
GROUP BY u.Race, v.Channel2
QUALIFY channel_rank <= 5
ORDER BY u.Race, channel_rank;

-- Top Channels by Gender
SELECT 
    u.Gender,
    v.Channel2,
    COUNT(*) AS session_count,
    COUNT(DISTINCT v.UserID0) AS unique_viewers,
    ROUND(AVG(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes,
    ROW_NUMBER() OVER (PARTITION BY u.Gender ORDER BY COUNT(*) DESC) AS channel_rank
FROM workspace.default.viewership v
LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
WHERE v.Channel2 IS NOT NULL AND u.Gender IS NOT NULL AND u.Gender != 'None'
GROUP BY u.Gender, v.Channel2
QUALIFY channel_rank <= 10
ORDER BY u.Gender, channel_rank;

-- Top Channels by Age Group
SELECT 
    CASE 
        WHEN u.Age < 18 THEN 'Under 18'
        WHEN u.Age BETWEEN 18 AND 24 THEN '18-24'
        WHEN u.Age BETWEEN 25 AND 34 THEN '25-34'
        WHEN u.Age BETWEEN 35 AND 44 THEN '35-44'
        WHEN u.Age BETWEEN 45 AND 54 THEN '45-54'
        WHEN u.Age >= 55 THEN '55+'
        ELSE 'Unknown'
    END AS age_group,
    v.Channel2,
    COUNT(*) AS session_count,
    COUNT(DISTINCT v.UserID0) AS unique_viewers,
    ROUND(AVG(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes,
    ROW_NUMBER() OVER (PARTITION BY 
        CASE 
            WHEN u.Age < 18 THEN 'Under 18'
            WHEN u.Age BETWEEN 18 AND 24 THEN '18-24'
            WHEN u.Age BETWEEN 25 AND 34 THEN '25-34'
            WHEN u.Age BETWEEN 35 AND 44 THEN '35-44'
            WHEN u.Age BETWEEN 45 AND 54 THEN '45-54'
            WHEN u.Age >= 55 THEN '55+'
            ELSE 'Unknown'
        END 
        ORDER BY COUNT(*) DESC) AS channel_rank
FROM workspace.default.viewership v
LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
WHERE v.Channel2 IS NOT NULL AND u.Age IS NOT NULL AND u.Age > 0
GROUP BY age_group, v.Channel2
QUALIFY channel_rank <= 5
ORDER BY 
    CASE age_group
        WHEN 'Under 18' THEN 1
        WHEN '18-24' THEN 2
        WHEN '25-34' THEN 3
        WHEN '35-44' THEN 4
        WHEN '45-54' THEN 5
        WHEN '55+' THEN 6
        ELSE 7
    END,
    channel_rank;

-- Top Channels by Province
SELECT 
    u.Province,
    v.Channel2,
    COUNT(*) AS session_count,
    COUNT(DISTINCT v.UserID0) AS unique_viewers,
    ROUND(AVG(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_duration_minutes,
    ROW_NUMBER() OVER (PARTITION BY u.Province ORDER BY COUNT(*) DESC) AS channel_rank
FROM workspace.default.viewership v
LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
WHERE v.Channel2 IS NOT NULL AND u.Province IS NOT NULL AND u.Province != 'None'
GROUP BY u.Province, v.Channel2
QUALIFY channel_rank <= 5
ORDER BY u.Province, channel_rank;

-- Demographic Summary: Viewing Behavior by Race and Gender
SELECT 
    u.Race,
    u.Gender,
    COUNT(DISTINCT v.UserID0) AS unique_viewers,
    COUNT(*) AS total_sessions,
    ROUND(AVG(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 60, 2) AS avg_session_minutes,
    ROUND(SUM(UNIX_TIMESTAMP(v.`Duration 2`) - UNIX_TIMESTAMP('1899-12-31 00:00:00')) / 3600, 2) AS total_viewing_hours
FROM workspace.default.viewership v
LEFT JOIN workspace.default.user_profile u ON v.UserID0 = u.UserID
WHERE u.Race IS NOT NULL AND u.Race != 'None' 
  AND u.Gender IS NOT NULL AND u.Gender != 'None'
GROUP BY u.Race, u.Gender
ORDER BY total_sessions DESC;
